In [3]:
import numpy as np
import random
from functools import reduce
from itertools import chain, combinations, product, repeat
from scipy import linalg, sparse
import matplotlib.pyplot as plt
import os
from scipy.stats import unitary_group
import time 
from toqito.channels import partial_trace

In [4]:
def PartialTrace(state, subsys):
    tensor_state = np.reshape(state, (2,)*int(np.log2(len(state))))
    # print(tensor_state.shape)
    rho_tensor = np.tensordot(tensor_state.conj(), tensor_state, axes=(subsys,subsys))
    subsys_dim = np.prod(rho_tensor.shape[:len(rho.shape)//2])
    return np.reshape(rho_tensor, (subsys_dim, subsys_dim))

In [5]:
def calcS(densityM,partiesToTraceOut=[1]):
    Ntot=int(np.log2(densityM.shape[0]))
    redRho = partial_trace(densityM, partiesToTraceOut, [2]* Ntot)
    eVals = np.linalg.eigvalsh(redRho)
    eVals = eVals[np.where(eVals>1e-12)]
    S = -sum(eVals*np.log2(eVals))
    return S

def calcPurity(densityM):
    purity = np.trace(densityM@densityM)
    return purity
    

In [6]:
def inner(X,Y):
    if type(X)!=np.ndarray:
        X=X.toarray()
        Y=Y.toarray()
    prod=np.trace(X.T.conj()@Y)
    return prod

In [57]:
def mkS_zMsm(probMsm, rho , debug=True):
    
    sites = [n for n in range(Nqubit)]
    randomNumbers = np.random.uniform(0,1,Nqubit)
    msmSites = np.where(randomNumbers < probMsm)[0]

    if list(msmSites) !=[]:

        Projs = [
            (np.eye(2)+S_Z)/2, 
            (np.eye(2)+(-1)*S_Z)/2
        ]

        numMsSites=len(msmSites)
        #print(msmSites)
        
        # make one projection msm at a time:
        for (j,ms) in enumerate(msmSites):  
            proj_chain = [np.eye(2)] * Nqubit
            proj_chain[ms] = Projs[0]
            proj0p = reduce(sparse.kron, proj_chain)
            
            proj_chain = [np.eye(2)] * Nqubit
            proj_chain[ms] = Projs[1]
            proj1p = reduce(sparse.kron, proj_chain)

            # probability of having one of the outcomes
            prob = np.trace((proj0p @ rho @ proj0p.T.conj()).toarray())
            if np.random.random(1) < prob:
                testRho = (proj0p @ rho @ proj0p.T.conj())/prob
                
            else:
                testRho = (proj1p @ rho @ proj1p.T.conj())/(1-prob)

            ## check if the other prob is complementary
            #prob1 = np.trace((proj1p@rho@proj1p.T.conj()).toarray())
            #print(prob+prob1)
            if debug: 
                print(np.trace(testRho.toarray()))
                print(proj_chain)
                
            # One needs to keep track of the measurement!!!
            rho = testRho

    else:
        testRho = rho
        print('no measure')

    return testRho 


In [10]:
# Gates:

Id2 = np.array([[1,0],[0,1]], dtype = 'complex128')
S_X = np.array([[0,1],[1,0]], dtype = 'complex128')
S_Y = np.array([[0,-1j],[1j,0]], dtype = 'complex128')
S_Z = np.array([[1,0],[0,-1]], dtype = 'complex128')
paulis = [Id2, S_X, S_Y, S_Z]/np.sqrt(2)
basis4by4=np.kron(paulis,paulis)


In [58]:
qb={}
qb[0]=np.array([1,0],dtype='complex128')
qb[1]=np.array([0,1],dtype='complex128')
rho0 = np.outer(qb[0], qb[0])
rho1 = np.outer(qb[1], qb[1])

Nqubit = 8

# pure state start
state0rho = (rho0 + rho1)/np.trace((rho0 + rho1))
rho = sparse.bsr_matrix(reduce(sparse.kron, [state0rho]*Nqubit))


#state0 = (qb[0] + qb[1])/np.sqrt(2)
#state0rho = np.outer(state0, state0)
#rho = sparse.bsr_matrix(reduce(sparse.kron, [state0rho]*Nqubit))


In [19]:
projOp=(np.eye(2)+S_Z)/2

(projOp @ rho @ projOp.T.conj())

array([[0.5+0.j, 0. +0.j],
       [0. +0.j, 0. +0.j]])

In [60]:
rho

<256x256 sparse matrix of type '<class 'numpy.complex128'>'
	with 32768 stored elements (blocksize = 2x2) in Block Sparse Row format>

In [69]:
test_out=rho
for i in range(10):
    print('purity=',calcPurity(test_out.toarray()))
    test_out = mkS_zMsm(0, test_out,debug=False)
    #print(test_out)
    print('S=',calcS(test_out.toarray(),[1,2,3,4]))


purity= (0.00390625+0j)
no measure
S= 4.0
purity= (0.00390625+0j)
no measure
S= 4.0
purity= (0.00390625+0j)
no measure
S= 4.0
purity= (0.00390625+0j)
no measure
S= 4.0
purity= (0.00390625+0j)
no measure
S= 4.0
purity= (0.00390625+0j)
no measure
S= 4.0
purity= (0.00390625+0j)
no measure
S= 4.0
purity= (0.00390625+0j)
no measure
S= 4.0
purity= (0.00390625+0j)
no measure
S= 4.0
purity= (0.00390625+0j)
no measure
S= 4.0


In [ ]:
calcPurity(rho.toarray())

In [ ]:
probMsm=0.5
sites = [n for n in range(Nqubit)]
randomNumbers = np.random.random(Nqubit)
msmSites = np.where(randomNumbers < probMsm)[0]


In [ ]:
Projs = [
        (np.eye(2)+S_Z)/2, 
        (np.eye(2)+(-1)*S_Z)/2
    ]

proj0p = Projs[0]
#prob = np.trace((proj0p @ state0rho @ proj0p.T.conj())) 
#testRho = (proj0p @ state0rho @ proj0p.T.conj())/prob

proj1p = Projs[1]
testRho = (proj1p @ state0rho @ proj1p.T.conj())/(1-prob)


In [12]:
rho

<2x2 sparse matrix of type '<class 'numpy.complex128'>'
	with 2 stored elements (blocksize = 1x1) in Block Sparse Row format>

In [ ]:
test_out=rho
for i in range(10):
    test_out = mkS_zMsm(1, test_out)
    print(calcS(test_out.toarray(),[1,2,3,4,5]))
    print(calcPurity(test_out.toarray()))

In [ ]:
sites = [n for n in range(Nqubit)]
randomNumbers = np.random.random(Nqubit)
msmSites = np.where(randomNumbers < probMsm)[0]

if list(msmSites) !=[]:

    Projs = [
        (np.eye(2)+S_Z)/2, 
        (np.eye(2)+(-1)*S_Z)/2
    ]

    numMsSites=len(msmSites)

    # make one projection msm at a time:
    for (j,ms) in enumerate(msmSites):  
        proj_chain = [np.eye(2)] * Nqubit
        proj_chain[ms] = Projs[0]
        proj0p = reduce(sparse.kron, proj_chain)

        proj_chain = [np.eye(2)] * Nqubit
        proj_chain[ms] = Projs[1]
        proj1p = reduce(sparse.kron, proj_chain)

        # probability of having one of the outcomes
        prob = np.trace((proj0p @ rho @ proj0p.T.conj()).toarray()) 
        if np.random.random(1) < prob:
            testRho = (proj0p @ rho @ proj0p.T.conj())/prob

        else:
            testRho = (proj1p @ rho @ proj1p.T.conj())/(1-prob)

        ## check if the other prob is complementary
        #prob1 = np.trace((proj1p@rho@proj1p.T.conj()).toarray())
        #print(prob+prob1)

else:
    testRho = rho


In [ ]:
test_out.toarray()